# OW-DETR Task 1 Training Notebook (IP102 - 25 Classes Split)
Notebook này thực thi **Task 1 (7 lớp)** cho bài toán Open World Object Detection (OW-DETR) trên bộ dữ liệu IP102 (25 lớp).
Sử dụng checkpoint Deformable DETR tại `/kaggle/input/models/chienkhu/deformable/pytorch/default/1/pytorch_model.bin`.

In [ ]:
import os

# 1. Quay về thư mục làm việc Kaggle
%cd /kaggle/working

if not os.path.exists("/kaggle/working/OW-DETR"):
    !git clone https://github.com/nta2112/OW-DETR.git /kaggle/working/OW-DETR
else:
    print("Thư mục OW-DETR đã tồn tại. Đang đồng bộ bản mới nhất...")
    %cd /kaggle/working/OW-DETR
    !git pull origin main

%cd /kaggle/working/OW-DETR


In [ ]:
%cd /kaggle/working/OW-DETR

# 2. Cài đặt các thư viện cần thiết
!pip install -r requirements.txt

# 3. Tải DINO ResNet-50 backbone pretrain
!mkdir -p models
!wget -N https://dl.fbaipublicfiles.com/dino/dino_resnet50_pretrain/dino_resnet50_pretrain.pth -O models/dino_resnet50_pretrain.pth

# 4. Biên dịch các CUDA operators (MultiScaleDeformableAttention)
%cd models/ops
!sh make.sh
!python test.py

%cd /kaggle/working/OW-DETR


In [ ]:
import torch
import os

# 5. Chuyển đổi trọng số từ checkpoint DETR (pytorch_model.bin) sang OW-DETR
input_path = "/kaggle/input/models/chienkhu/deformable/pytorch/default/1/pytorch_model.bin"
converted_path = "/kaggle/working/OW-DETR/models/deformable_detr_coco_converted.pth"

print("-> Đang ánh xạ trọng số DETR checkpoint tương thích với OW-DETR...")
os.makedirs("/kaggle/working/OW-DETR/models", exist_ok=True)

if not os.path.exists(converted_path):
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Không tìm thấy checkpoint tại {input_path}. Vui lòng kiểm tra Kaggle input.")
        
    state_dict = torch.load(input_path, map_location="cpu")
    new_state_dict = {}
    
    for k, v in state_dict.items():
        # Bỏ qua class_embed do số lượng lớp thay đổi
        if "class_embed." in k:
            continue
            
        new_k = k
        if k.startswith("model.backbone.conv_encoder.model."):
            new_k = k.replace("model.backbone.conv_encoder.model.", "backbone.0.body.")
        elif k.startswith("model.encoder."):
            new_k = k.replace("model.encoder.", "transformer.encoder.")
        elif k.startswith("model.decoder."):
            new_k = k.replace("model.decoder.", "transformer.decoder.")
        elif k == "model.level_embed":
            new_k = "transformer.level_embed"
        elif k.startswith("model.reference_points."):
            new_k = k.replace("model.reference_points.", "transformer.reference_points.")
        elif k == "model.query_position_embeddings.weight":
            new_k = "query_embed.weight"
            if v.shape[0] > 100:
                print(f"  [Đã cắt] query_embed.weight từ {list(v.shape)} xuống [100, 512]")
                v = v[:100, :]
        elif k.startswith("model."):
            new_k = k[6:]
            
        new_state_dict[new_k] = v

    torch.save({"model": new_state_dict, "epoch": -1}, converted_path)
    print(f"✓ Hoàn thành! Đã chuyển đổi checkpoint thành công: {converted_path}")
else:
    print("-> Checkpoint tương thích đã tồn tại, bỏ qua chuyển đổi.")


In [ ]:
import os
import glob
import shutil

# 6. Thiết lập thư mục dữ liệu VOC2007 và symlink ảnh/nhãn
voc_dir = "data/IP102/VOC2007"
os.makedirs(voc_dir, exist_ok=True)

def find_kaggle_dir(folder_name):
    candidates = [
        f"/kaggle/input/datasets/nta212/ip102-for-object-detection/VOC2007/{folder_name}",
        f"/kaggle/input/ip102-for-object-detection/VOC2007/{folder_name}",
        f"/kaggle/input/ip102-for-object-detection/{folder_name}"
    ]
    for cand in candidates:
        if os.path.isdir(cand):
            return cand
    matches = glob.glob(f"/kaggle/input/**/{folder_name}", recursive=True)
    for m in matches:
        if os.path.isdir(m):
            return m
    raise FileNotFoundError(f"Không tìm thấy thư mục {folder_name} trong /kaggle/input")

jpeg_src = find_kaggle_dir("JPEGImages")
annot_src = find_kaggle_dir("Annotations")

print(f"✓ Tìm thấy JPEGImages nguồn tại: {jpeg_src}")
print(f"✓ Tìm thấy Annotations nguồn tại: {annot_src}")

jpeg_target = os.path.join(voc_dir, "JPEGImages")
annot_target = os.path.join(voc_dir, "Annotations")

# Xóa symlink hỏng nếu có
if os.path.islink(jpeg_target) or os.path.exists(jpeg_target):
    if os.path.islink(jpeg_target):
        os.unlink(jpeg_target)
    elif os.path.isdir(jpeg_target):
        shutil.rmtree(jpeg_target)

if os.path.islink(annot_target) or os.path.exists(annot_target):
    if os.path.islink(annot_target):
        os.unlink(annot_target)
    elif os.path.isdir(annot_target):
        shutil.rmtree(annot_target)

os.symlink(jpeg_src, jpeg_target)
os.symlink(annot_src, annot_target)

print(f"✓ Đã kết nối symlink thành công: {jpeg_target} -> {jpeg_src}")
print(f"✓ Đã kết nối symlink thành công: {annot_target} -> {annot_src}")

# Kiểm tra thử 1 file ảnh có mở được không
test_file = os.path.join(jpeg_target, "001536.jpg")
if os.path.exists(test_file):
    print(f"✓ Xác nhận file ảnh thử nghiệm tồn tại: {test_file}")
else:
    sample_files = os.listdir(jpeg_target)[:3] if os.path.exists(jpeg_target) else []
    print(f"⚠️ Mẫu 3 ảnh thực tế có trong thư mục: {sample_files}")


In [ ]:
import os
import glob
from pycocotools.coco import COCO

# 7. Chia dữ liệu theo cấu hình mới (Task 1 = 7 lớp: chỉ số 0..6)
def find_kaggle_file(file_name):
    candidates = [
        f"/kaggle/input/datasets/nta212/ip102-for-object-detection/{file_name}",
        f"/kaggle/input/ip102-for-object-detection/{file_name}"
    ]
    for cand in candidates:
        if os.path.exists(cand):
            return cand
    matches = glob.glob(f"/kaggle/input/**/{file_name}", recursive=True)
    if matches:
        return matches[0]
    return candidates[0]

def find_kaggle_dir(folder_name):
    candidates = [
        f"/kaggle/input/datasets/nta212/ip102-for-object-detection/VOC2007/{folder_name}",
        f"/kaggle/input/ip102-for-object-detection/VOC2007/{folder_name}",
        f"/kaggle/input/ip102-for-object-detection/{folder_name}",
        f"data/IP102/VOC2007/{folder_name}"
    ]
    for cand in candidates:
        if os.path.isdir(cand):
            return cand
    matches = glob.glob(f"/kaggle/input/**/{folder_name}", recursive=True)
    for m in matches:
        if os.path.isdir(m):
            return m
    return None

json_path = find_kaggle_file("train.json")
print(f"✓ Đã tìm thấy file train.json tại: {json_path}")
coco = COCO(json_path)

voc_img_dir = find_kaggle_dir("JPEGImages")
existing_stems = set()
if voc_img_dir and os.path.exists(voc_img_dir):
    for f in os.listdir(voc_img_dir):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            existing_stems.add(os.path.splitext(f)[0])
    print(f"✓ Đã quét thành công {len(existing_stems)} file ảnh thực tế từ: {voc_img_dir}")
else:
    print("⚠️ Chưa tìm thấy JPEGImages nguồn, dùng định dạng chuẩn 6 chữ số.")

def get_valid_stem(img_id):
    img_info = coco.imgs.get(img_id, {})
    file_name = img_info.get("file_name", "")
    fn_stem = os.path.splitext(os.path.basename(file_name))[0] if file_name else ""
    
    candidates = []
    if str(img_id).isdigit():
        val = int(img_id)
        short_val = val % 1000000
        candidates.append(f"{short_val:06d}")
        candidates.append(f"{val:06d}")
        candidates.append(str(val))
    if fn_stem:
        candidates.append(fn_stem)

    if existing_stems:
        for cand in candidates:
            if cand in existing_stems:
                return cand
    return candidates[0] if candidates else str(img_id)

# Sắp xếp danh mục theo ID để lấy đúng 7 lớp đầu tiên cho Task 1
cats = sorted(coco.loadCats(coco.getCatIds()), key=lambda c: c['id'])

task_cat_ids = {
    't1': set([c['id'] for c in cats[0:7]]),
    't2': set([c['id'] for c in cats[7:13]]),
    't3': set([c['id'] for c in cats[13:19]]),
    't4': set([c['id'] for c in cats[19:25]])
}

train_images = {}
for task_name, cat_ids in task_cat_ids.items():
    task_img_ids = set()
    for ann in coco.anns.values():
        if ann['category_id'] in cat_ids:
            task_img_ids.add(ann['image_id'])
    train_images[task_name] = sorted(list(task_img_ids))

all_img_ids = sorted(list(coco.imgs.keys()))
output_dir = "data/IP102/VOC2007/ImageSets/Main"
os.makedirs(output_dir, exist_ok=True)

splits = {
    "t1_train": train_images["t1"],
    "val": all_img_ids,
    "test": all_img_ids,
}

for name, img_ids in splits.items():
    path = os.path.join(output_dir, f"{name}.txt")
    valid_stems = [get_valid_stem(img_id) for img_id in img_ids]
    with open(path, "w") as f:
        for stem in valid_stems:
            f.write(f"{stem}\n")
    print(f"Đã tạo file {name}.txt với {len(valid_stems)} ảnh. Mẫu tên file: {valid_stems[:3]}")


In [ ]:
%cd /kaggle/working/OW-DETR

# 8. Tiến hành huấn luyện Task 1 (7 lớp)
!torchrun --nproc_per_node=2 main_open_world.py \
    --output_dir '/kaggle/working/exps/ip102_t1' \
    --dataset owod \
    --dec_layers 6 \
    --num_queries 100 \
    --batch_size 2 \
    --PREV_INTRODUCED_CLS 0 \
    --CUR_INTRODUCED_CLS 7 \
    --data_root 'data/IP102' \
    --train_set 't1_train' \
    --val_set 'val' \
    --test_set 'test' \
    --num_classes 26 \
    --unmatched_boxes \
    --top_unk 5 \
    --featdim 1024 \
    --NC_branch \
    --backbone 'dino_resnet50' \
    --pretrain 'models/deformable_detr_coco_converted.pth' \
    --num_workers 4 \
    --eval_every 1 \
    --cache_mode \
    --epochs 51


In [ ]:
%cd /kaggle/working/OW-DETR

# 9. Đánh giá (Evaluate) mô hình Task 1 sau khi huấn luyện
!torchrun --nproc_per_node=2 main_open_world.py \
    --dataset owod \
    --dec_layers 6 \
    --num_queries 100 \
    --batch_size 10 \
    --PREV_INTRODUCED_CLS 0 \
    --CUR_INTRODUCED_CLS 7 \
    --data_root 'data/IP102' \
    --train_set 't1_train' \
    --val_set 'val' \
    --test_set 'test' \
    --num_classes 26 \
    --unmatched_boxes \
    --top_unk 5 \
    --featdim 1024 \
    --NC_branch \
    --backbone 'dino_resnet50' \
    --resume '/kaggle/working/exps/ip102_t1/best_checkpoint.pth' \
    --eval


In [ ]:
import shutil
import os

# 10. Nén toàn bộ checkpoint và log của Task 1 thành file zip để tải về
source_dir = "/kaggle/working/exps/ip102_t1"
output_zip = "/kaggle/working/ip102_t1"

if os.path.exists(source_dir):
    print(f"-> Đang nén thư mục checkpoint {source_dir}...")
    shutil.make_archive(output_zip, "zip", source_dir)
    zip_file = output_zip + ".zip"
    size_mb = os.path.getsize(zip_file) / (1024 * 1024)
    print(f"✓ Hoàn thành! File zip được lưu tại: {zip_file} ({size_mb:.2f} MB)")
else:
    print(f"Không tìm thấy thư mục {source_dir}. Vui lòng chạy cell huấn luyện Task 1 trước!")
